In [4]:
import pandas as pd
import numpy as np

# Load features
df = pd.read_csv(
    r"C:\Users\HP\Desktop\MSIMI_Project\data\features\08_patient_phase_features.csv"
)

# Keep 500 Hz data
df = df[df["sampling_rate"] == 500].copy()

# Load clinical labels
import os, json

folder = r"C:\Users\HP\Downloads\MSIMI_DB"
labels = []

for file in os.listdir(folder):
    if file.endswith("_disease.json"):
        sid = file.replace("_disease.json", "")
        with open(os.path.join(folder, file), encoding="utf-8") as f:
            data = json.load(f)
        labels.append({
            "subject_id": sid,
            "target": data.get("MSIMI-based-ecg")
        })

labels_df = pd.DataFrame(labels)
labels_df["target"] = pd.to_numeric(
    labels_df["target"], errors="coerce"
)

# Merge labels
df = df.merge(labels_df, on="subject_id", how="left")

# Development patients only
df = df[
    (df["is_unlabelled_holdout"] == False) &
    (df["target"].notna())
].copy()

print("Dataset:", df.shape)
print(df["target"].value_counts())

Dataset: (78, 265)
target
0.0    69
1.0     9
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np

# Load features
df = pd.read_csv(
    r"C:\Users\HP\Desktop\MSIMI_Project\data\features\08_patient_phase_features.csv"
)

# Keep 500 Hz data
df = df[df["sampling_rate"] == 500].copy()

# Load clinical labels
import os, json

folder = r"C:\Users\HP\Downloads\MSIMI_DB"
labels = []

for file in os.listdir(folder):
    if file.endswith("_disease.json"):
        sid = file.replace("_disease.json", "")
        with open(os.path.join(folder, file), encoding="utf-8") as f:
            data = json.load(f)
        labels.append({
            "subject_id": sid,
            "target": data.get("MSIMI-based-ecg")
        })

labels_df = pd.DataFrame(labels)
labels_df["target"] = pd.to_numeric(
    labels_df["target"], errors="coerce"
)

# Merge labels
df = df.merge(labels_df, on="subject_id", how="left")

# Development patients only
df = df[
    (df["is_unlabelled_holdout"] == False) &
    (df["target"].notna())
].copy()

print("Dataset:", df.shape)
print(df["target"].value_counts())

Dataset: (78, 265)
target
0.0    69
1.0     9
Name: count, dtype: int64


In [6]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Remove metadata and target
metadata = [
    "subject_id",
    "sampling_rate",
    "is_unlabelled_holdout",
    "stress_recording_status",
    "stage06_candidate_nn_status",
    "target"
]

X = df.drop(columns=metadata, errors="ignore")
y = df["target"].astype(int)

# Temporarily exclude ST-related features
st_pattern = re.compile(r"st60|st80|st_slope|min_st", re.I)
X = X.loc[:, ~X.columns.str.contains(st_pattern)]

# Remove completely missing and constant columns
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

print("Patients:", len(X))
print("Candidate features:", X.shape[1])

# Models
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():

    predictions = cross_val_predict(
        model, X, y, cv=cv, method="predict"
    )

    probabilities = cross_val_predict(
        model, X, y, cv=cv, method="predict_proba"
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Sensitivity": recall_score(y, predictions, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0,
        "F1 Score": f1_score(y, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y, probabilities)
    })

results_df = pd.DataFrame(results)
print("\nMODEL COMPARISON")
display(results_df.round(4))

Patients: 78
Candidate features: 86

MODEL COMPARISON


,Model,Accuracy,Balanced Accuracy,Precision,Sensitivity,Specificity,F1 Score,ROC-AUC
0,Logistic Regression,0.7692,0.4831,0.0909,0.1111,0.8551,0.1,0.4251
1,Random Forest,0.8462,0.4783,0.0000,0.0000,0.9565,0.0,0.5346


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

for name, model in models.items():

    pred = cross_val_predict(
        model, X, y, cv=cv, method="predict"
    )

    prob = cross_val_predict(
        model, X, y, cv=cv, method="predict_proba"
    )[:, 1]

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, pred))

    print("\nClassification Report:")
    print(classification_report(
        y, pred, zero_division=0
    ))

    print("\nPositive probability range:")
    print("Minimum:", round(prob.min(), 4))
    print("Maximum:", round(prob.max(), 4))
    print("Mean:", round(prob.mean(), 4))


Logistic Regression

Confusion Matrix:
[[59 10]
 [ 8  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87        69
           1       0.09      0.11      0.10         9

    accuracy                           0.77        78
   macro avg       0.49      0.48      0.48        78
weighted avg       0.79      0.77      0.78        78


Positive probability range:
Minimum: 0.0
Maximum: 0.9981
Mean: 0.2196

Random Forest

Confusion Matrix:
[[66  3]
 [ 9  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92        69
           1       0.00      0.00      0.00         9

    accuracy                           0.85        78
   macro avg       0.44      0.48      0.46        78
weighted avg       0.78      0.85      0.81        78


Positive probability range:
Minimum: 0.0367
Maximum: 0.5733
Mean: 0.2451


In [8]:
rf = models["Random Forest"]

rf_pred = cross_val_predict(
    rf, X, y, cv=cv, method="predict"
)

rf_prob = cross_val_predict(
    rf, X, y, cv=cv, method="predict_proba"
)[:, 1]

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y, rf_pred))

print("\nClassification Report:")
print(classification_report(y, rf_pred, zero_division=0))

print("\nPositive patients' predicted probabilities:")
print(
    pd.DataFrame({
        "Actual": y.values,
        "Probability": rf_prob
    }).query("Actual == 1").round(4)
)

Random Forest Confusion Matrix:
[[66  3]
 [ 9  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92        69
           1       0.00      0.00      0.00         9

    accuracy                           0.85        78
   macro avg       0.44      0.48      0.46        78
weighted avg       0.78      0.85      0.81        78


Positive patients' predicted probabilities:
    Actual  Probability
0        1       0.0400
2        1       0.1533
5        1       0.3467
23       1       0.4767
24       1       0.4733
28       1       0.3167
32       1       0.2500
33       1       0.1033
64       1       0.1900


In [9]:
from sklearn.metrics import confusion_matrix

print("RANDOM FOREST THRESHOLD ANALYSIS")

for threshold in [0.50, 0.40, 0.30, 0.20, 0.10]:

    pred = (rf_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print(f"\nThreshold: {threshold}")
    print("TP:", tp, "FP:", fp)
    print("TN:", tn, "FN:", fn)
    print("Sensitivity:", round(sensitivity, 3))
    print("Specificity:", round(specificity, 3))

RANDOM FOREST THRESHOLD ANALYSIS

Threshold: 0.5
TP: 0 FP: 3
TN: 66 FN: 9
Sensitivity: 0.0
Specificity: 0.957

Threshold: 0.4
TP: 2 FP: 8
TN: 61 FN: 7
Sensitivity: 0.222
Specificity: 0.884

Threshold: 0.3
TP: 4 FP: 25
TN: 44 FN: 5
Sensitivity: 0.444
Specificity: 0.638

Threshold: 0.2
TP: 5 FP: 43
TN: 26 FN: 4
Sensitivity: 0.556
Specificity: 0.377

Threshold: 0.1
TP: 8 FP: 57
TN: 12 FN: 1
Sensitivity: 0.889
Specificity: 0.174


In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score

# Random Forest AP
rf_ap = average_precision_score(y, rf_prob)

# Dummy baseline
dummy = DummyClassifier(strategy="prior")

dummy_prob = cross_val_predict(
    dummy,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

dummy_ap = average_precision_score(y, dummy_prob)

print("========== BASELINE COMPARISON ==========")
print("Random Forest AP:", round(rf_ap, 4))
print("Dummy Baseline AP:", round(dummy_ap, 4))
print("Positive Class Prevalence:", round(y.mean(), 4))

========== BASELINE COMPARISON ==========
Random Forest AP: 0.1776
Dummy Baseline AP: 0.1154
Positive Class Prevalence: 0.1154


In [11]:
# Feature audit
feature_audit = pd.DataFrame({
    "Feature": X.columns,
    "Missing_Count": X.isna().sum().values,
    "Unique_Values": X.nunique().values
})

# Positive and negative patients ka feature comparison
positive_mean = X[y == 1].mean()
negative_mean = X[y == 0].mean()

feature_audit["Positive_Mean"] = positive_mean.values
feature_audit["Negative_Mean"] = negative_mean.values

feature_audit["Mean_Difference"] = (
    feature_audit["Positive_Mean"] -
    feature_audit["Negative_Mean"]
)

display(
    feature_audit.sort_values(
        "Mean_Difference",
        key=lambda s: s.abs(),
        ascending=False
    ).round(3)
)

print("Total features:", len(X.columns))
print("\nAll feature names:")
for i, name in enumerate(X.columns, 1):
    print(i, name)

,Feature,Missing_Count,Unique_Values,Positive_Mean,Negative_Mean,Mean_Difference
79,t1_frobenius_dissimilarity_adc_norm,0,78,1849.729,1461.265,388.464
1,rest_raw_detector_rr_rmssd_ms,0,78,143.877,260.187,-116.310
22,rest_V5_t_amp_adc,0,78,175.706,68.288,107.418
53,recover_raw_detector_rr_rmssd_ms,0,78,125.100,228.880,-103.780
84,t2_frobenius_dissimilarity_adc_norm,0,78,1258.435,1163.120,95.315
...,...,...,...,...,...,...
9,rest_aVR_qrs_dur_ms,0,32,60.222,59.768,0.454
11,rest_aVL_qrs_dur_ms,0,43,85.556,85.275,0.280
80,t1_mean_morphology_corr,0,78,0.692,0.756,-0.065
85,t2_mean_morphology_corr,0,78,0.773,0.834,-0.061


Total features: 86

All feature names:
1 rest_qrs_axis_degrees
2 rest_raw_detector_rr_rmssd_ms
3 rest_I_t_amp_adc
4 rest_I_qrs_dur_ms
5 rest_II_t_amp_adc
6 rest_II_qrs_dur_ms
7 rest_III_t_amp_adc
8 rest_III_qrs_dur_ms
9 rest_aVR_t_amp_adc
10 rest_aVR_qrs_dur_ms
11 rest_aVL_t_amp_adc
12 rest_aVL_qrs_dur_ms
13 rest_aVF_t_amp_adc
14 rest_aVF_qrs_dur_ms
15 rest_V1_t_amp_adc
16 rest_V1_qrs_dur_ms
17 rest_V2_t_amp_adc
18 rest_V2_qrs_dur_ms
19 rest_V3_t_amp_adc
20 rest_V3_qrs_dur_ms
21 rest_V4_t_amp_adc
22 rest_V4_qrs_dur_ms
23 rest_V5_t_amp_adc
24 rest_V5_qrs_dur_ms
25 rest_V6_t_amp_adc
26 rest_V6_qrs_dur_ms
27 stress_qrs_axis_degrees
28 stress_raw_detector_rr_rmssd_ms
29 stress_I_t_amp_adc
30 stress_I_qrs_dur_ms
31 stress_II_t_amp_adc
32 stress_II_qrs_dur_ms
33 stress_III_t_amp_adc
34 stress_III_qrs_dur_ms
35 stress_aVR_t_amp_adc
36 stress_aVR_qrs_dur_ms
37 stress_aVL_t_amp_adc
38 stress_aVL_qrs_dur_ms
39 stress_aVF_t_amp_adc
40 stress_aVF_qrs_dur_ms
41 stress_V1_t_amp_adc
42 stress_V1_qrs_

In [12]:
# Missing-value audit
missing = X.isna().sum()
print("Features with missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Group-wise feature count
groups = {
    "Rest": [c for c in X.columns if c.startswith("rest_")],
    "Stress": [c for c in X.columns if c.startswith("stress_")],
    "Recovery": [c for c in X.columns if c.startswith("recover_")],
    "T1": [c for c in X.columns if c.startswith("t1_")],
    "T2": [c for c in X.columns if c.startswith("t2_")]
}

print("\nFeature Groups:")
for name, features in groups.items():
    print(name, ":", len(features))

Features with missing values:
t2_hrri_pct    65
dtype: int64

Feature Groups:
Rest : 26
Stress : 26
Recovery : 26
T1 : 3
T2 : 5


train-test-split + model accuracy

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score
)

# Remove feature with excessive missing values
X_model = X.drop(columns=["t2_hrri_pct"], errors="ignore")

# 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training patients:", len(X_train))
print("Testing patients:", len(X_test))
print("Training positive cases:", sum(y_train))
print("Testing positive cases:", sum(y_test))

# Train and evaluate both models
for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 45)
    print(name)
    print("=" * 45)

    print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

Training patients: 62
Testing patients: 16
Training positive cases: 7
Testing positive cases: 2

Logistic Regression
Accuracy: 75.0 %

Confusion Matrix:
[[12  2]
 [ 2  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86        14
           1       0.00      0.00      0.00         2

    accuracy                           0.75        16
   macro avg       0.43      0.43      0.43        16
weighted avg       0.75      0.75      0.75        16

ROC-AUC: 0.0714

Random Forest
Accuracy: 87.5 %

Confusion Matrix:
[[14  0]
 [ 2  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93        14
           1       0.00      0.00      0.00         2

    accuracy                           0.88        16
   macro avg       0.44      0.50      0.47        16
weighted avg       0.77      0.88      0.82        16

ROC-AUC: 0.5714


In [14]:
print("Total patients:", len(df))

print("\nHoldout status:")
print(df["is_unlabelled_holdout"].value_counts())

print("\nMissing clinical labels:")
print(df["target"].isna().sum())

Total patients: 78

Holdout status:
is_unlabelled_holdout
False    78
Name: count, dtype: int64

Missing clinical labels:
0


In [15]:
import pandas as pd

df_all = pd.read_csv(
    r"C:\Users\HP\Desktop\MSIMI_Project\data\features\08_patient_phase_features.csv"
)

df_500 = df_all[df_all["sampling_rate"] == 500].copy()

print("Total 500 Hz patients:", len(df_500))
print("\nHoldout status:")
print(df_500["is_unlabelled_holdout"].value_counts())

Total 500 Hz patients: 80

Holdout status:
is_unlabelled_holdout
False    78
True      2
Name: count, dtype: int64


In [17]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, roc_auc_score

from xgboost import XGBClassifier

# Prepare data
X_model = X.drop(columns=["t2_hrri_pct"], errors="ignore").copy()
X_model = X_model.reset_index(drop=True)
y_model = pd.Series(y).reset_index(drop=True)

# Feature groups
groups = {
    "Complete": X_model.columns.tolist(),
    "Rest": [c for c in X_model.columns if c.lower().startswith("rest_")],
    "Stress": [c for c in X_model.columns if c.lower().startswith("stress_")],
    "Recover": [c for c in X_model.columns if c.lower().startswith("recover_")]
}

groups["Rest + Stress + Recover"] = (
    groups["Rest"] + groups["Stress"] + groups["Recover"]
)

print("Feature counts:")
for name, cols in groups.items():
    print(name, ":", len(cols))

# Models
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ),
    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),
    "Kneighbors": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        scale_pos_weight=69/9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        tree_method="hist"
    )
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

results = []

# Sequential cross-validation
for group_name, columns in groups.items():

    print("\nProcessing:", group_name, flush=True)

    X_group = X_model[columns]
    oof_prob = np.zeros(len(y_model))

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X_group, y_model), start=1
    ):

        X_train = X_group.iloc[train_idx]
        X_test = X_group.iloc[test_idx]
        y_train = y_model.iloc[train_idx]

        for model_name, estimator in models.items():

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", clone(estimator))
            ])

            pipeline.fit(X_train, y_train)

            # Store out-of-fold probabilities
            probabilities = pipeline.predict_proba(X_test)[:, 1]

            if fold == 1:
                oof_prob = np.zeros(
                    (len(models), len(y_model))
                )

            model_index = list(models.keys()).index(model_name)
            oof_prob[model_index, test_idx] = probabilities

    for model_index, model_name in enumerate(models.keys()):

        probabilities = oof_prob[model_index]
        predictions = (probabilities >= 0.5).astype(int)

        results.append({
            "Dataset": group_name,
            "Model": model_name,
            "ROC-AUC (%)": round(
                roc_auc_score(y_model, probabilities) * 100, 2
            ),
            "Accuracy (%)": round(
                accuracy_score(y_model, predictions) * 100, 2
            )
        })

        print(
            model_name,
            "| Accuracy:", round(
                accuracy_score(y_model, predictions) * 100, 2
            ),
            "| ROC-AUC:", round(
                roc_auc_score(y_model, probabilities) * 100, 2
            ),
            flush=True
        )

# Create final comparison table
results_df = pd.DataFrame(results)

table = results_df.pivot(
    index="Dataset",
    columns="Model",
    values=["ROC-AUC (%)", "Accuracy (%)"]
)

model_order = list(models.keys())

table = table.reindex(columns=pd.MultiIndex.from_product([
    ["ROC-AUC (%)", "Accuracy (%)"],
    model_order
]))

display(table)

# Save CSV
results_df.to_csv(
    "MSIMI_Model_Comparison_Results.csv",
    index=False
)

print("\nComparison completed successfully!")


Feature counts:
Complete : 85
Rest : 26
Stress : 26
Recover : 26
Rest + Stress + Recover : 78

Processing: Complete


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Logistic Regression | Accuracy: 80.77 | ROC-AUC: 46.94
SVM | Accuracy: 88.46 | ROC-AUC: 58.21
Random Forest | Accuracy: 83.33 | ROC-AUC: 51.93
Kneighbors | Accuracy: 88.46 | ROC-AUC: 44.93
XGBoost | Accuracy: 85.9 | ROC-AUC: 53.62

Processing: Rest


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Logistic Regression | Accuracy: 78.21 | ROC-AUC: 40.34
SVM | Accuracy: 88.46 | ROC-AUC: 55.64
Random Forest | Accuracy: 85.9 | ROC-AUC: 53.62
Kneighbors | Accuracy: 88.46 | ROC-AUC: 49.03
XGBoost | Accuracy: 84.62 | ROC-AUC: 73.03

Processing: Stress


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


... (5 outputs left out)

c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Logistic Regression | Accuracy: 78.21 | ROC-AUC: 46.78
SVM | Accuracy: 88.46 | ROC-AUC: 59.02
Random Forest | Accuracy: 87.18 | ROC-AUC: 44.69
Kneighbors | Accuracy: 88.46 | ROC-AUC: 45.65
XGBoost | Accuracy: 84.62 | ROC-AUC: 60.14


ROC-AUC (%)                                  \
                        Logistic Regression    SVM Random Forest Kneighbors   
Dataset                                                                       
Complete                              46.94  58.21         51.93      44.93   
Recover                               46.14  51.29         35.19      40.58   
Rest                                  40.34  55.64         53.62      49.03   
Rest + Stress + Recover               46.78  59.02         44.69      45.65   
Stress                                48.07  59.98         43.24      40.74   

                                       Accuracy (%)                       \
                        XGBoost Logistic Regression    SVM Random Forest   
Dataset                                                                    
Complete                  53.62               80.77  88.46         83.33   
Recover                   35.43               74.36  88.46         80.77   
Rest                      73.03               78.21  88.46         85.90   
Rest + Stress + Recover   60.14               78.21  88.46         87.18   
Stress                    46.54               76.92  88.46         80.77   

                                            
                        Kneighbors XGBoost  
Dataset                                     
Complete                     88.46   85.90  
Recover                      88.46   76.92  
Rest                         88.46   84.62  
Rest + Stress + Recover      88.46   84.62  
Stress                       87.18   74.36


Comparison completed successfully!


In [18]:

from sklearn.metrics import confusion_matrix, recall_score

diagnostics = []

for group_name, columns in groups.items():

    X_group = X_model[columns]

    for model_name, estimator in models.items():

        oof_prob = np.zeros(len(y_model))

        for train_idx, test_idx in cv.split(X_group, y_model):

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", clone(estimator))
            ])

            pipeline.fit(
                X_group.iloc[train_idx],
                y_model.iloc[train_idx]
            )

            oof_prob[test_idx] = pipeline.predict_proba(
                X_group.iloc[test_idx]
            )[:, 1]

        pred = (oof_prob >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_model, pred, labels=[0, 1]
        ).ravel()

        diagnostics.append({
            "Dataset": group_name,
            "Model": model_name,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "Sensitivity (%)": round(tp / (tp + fn) * 100, 2),
            "Specificity (%)": round(tn / (tn + fp) * 100, 2)
        })

diagnostics_df = pd.DataFrame(diagnostics)
display(diagnostics_df)

diagnostics_df.to_csv(
    "MSIMI_Confusion_Matrix_Metrics.csv",
    index=False
)


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

,Dataset,Model,TP,FP,TN,FN,Sensitivity (%),Specificity (%)
0,Complete,Logistic Regression,1,11,58,8,11.11,84.06
1,Complete,SVM,0,0,69,9,0.00,100.00
2,Complete,Random Forest,1,5,64,8,11.11,92.75
3,Complete,Kneighbors,0,0,69,9,0.00,100.00
4,Complete,XGBoost,2,4,65,7,22.22,94.20
5,Rest,Logistic Regression,1,12,57,8,11.11,82.61
6,Rest,SVM,0,1,68,9,0.00,98.55
7,Rest,Random Forest,1,4,65,8,11.11,94.20
8,Rest,Kneighbors,0,0,69,9,0.00,100.00
9,Rest,XGBoost,3,6,63,6,33.33,91.30


In [19]:

for group_name, columns in groups.items():

    print("\n" + "=" * 60)
    print(group_name, "| Total:", len(columns))
    print("=" * 60)

    for i, feature in enumerate(columns, start=1):
        print(f"{i}. {feature}")

# Extra temporal features
temporal_features = [
    c for c in X_model.columns
    if c.lower().startswith(("t1_", "t2_"))
]

print("\nTEMPORAL FEATURES")
for feature in temporal_features:
    print(feature)



Complete | Total: 85
1. rest_qrs_axis_degrees
2. rest_raw_detector_rr_rmssd_ms
3. rest_I_t_amp_adc
4. rest_I_qrs_dur_ms
5. rest_II_t_amp_adc
6. rest_II_qrs_dur_ms
7. rest_III_t_amp_adc
8. rest_III_qrs_dur_ms
9. rest_aVR_t_amp_adc
10. rest_aVR_qrs_dur_ms
11. rest_aVL_t_amp_adc
12. rest_aVL_qrs_dur_ms
13. rest_aVF_t_amp_adc
14. rest_aVF_qrs_dur_ms
15. rest_V1_t_amp_adc
16. rest_V1_qrs_dur_ms
17. rest_V2_t_amp_adc
18. rest_V2_qrs_dur_ms
19. rest_V3_t_amp_adc
20. rest_V3_qrs_dur_ms
21. rest_V4_t_amp_adc
22. rest_V4_qrs_dur_ms
23. rest_V5_t_amp_adc
24. rest_V5_qrs_dur_ms
25. rest_V6_t_amp_adc
26. rest_V6_qrs_dur_ms
27. stress_qrs_axis_degrees
28. stress_raw_detector_rr_rmssd_ms
29. stress_I_t_amp_adc
30. stress_I_qrs_dur_ms
31. stress_II_t_amp_adc
32. stress_II_qrs_dur_ms
33. stress_III_t_amp_adc
34. stress_III_qrs_dur_ms
35. stress_aVR_t_amp_adc
36. stress_aVR_qrs_dur_ms
37. stress_aVL_t_amp_adc
38. stress_aVL_qrs_dur_ms
39. stress_aVF_t_amp_adc
40. stress_aVF_qrs_dur_ms
41. stress_V1_t_a

85 ECG features mein new Personalized Temporal Features add karenge.

In [20]:

import numpy as np
import pandas as pd

# Copy existing 85 features
X_base = X_model.copy()

# Find matching features across all three phases
rest_cols = {
    c.replace("rest_", "", 1): c
    for c in X_base.columns if c.startswith("rest_")
}

stress_cols = {
    c.replace("stress_", "", 1): c
    for c in X_base.columns if c.startswith("stress_")
}

recover_cols = {
    c.replace("recover_", "", 1): c
    for c in X_base.columns if c.startswith("recover_")
}

common_features = (
    set(rest_cols) & set(stress_cols) & set(recover_cols)
)

print("Common phase features:", len(common_features))

# Create new temporal features
X_temporal = pd.DataFrame(index=X_base.index)

for feature in sorted(common_features):

    rest = pd.to_numeric(
        X_base[rest_cols[feature]], errors="coerce"
    )

    stress = pd.to_numeric(
        X_base[stress_cols[feature]], errors="coerce"
    )

    recover = pd.to_numeric(
        X_base[recover_cols[feature]], errors="coerce"
    )

    # Stress response
    X_temporal[f"temporal_stress_minus_rest_{feature}"] = (
        stress - rest
    )

    # Recovery change
    X_temporal[f"temporal_recovery_minus_stress_{feature}"] = (
        recover - stress
    )

    # Final recovery gap from baseline
    X_temporal[f"temporal_recovery_minus_rest_{feature}"] = (
        recover - rest
    )

    # Relative stress change (%)
    denominator_rest = rest.abs().replace(0, np.nan)

    X_temporal[f"temporal_stress_change_pct_{feature}"] = (
        (stress - rest) / denominator_rest
    ) * 100

    # Relative recovery change (%)
    denominator_stress = stress.abs().replace(0, np.nan)

    X_temporal[f"temporal_recovery_change_pct_{feature}"] = (
        (recover - stress) / denominator_stress
    ) * 100

# Combine original and engineered features
X_augmented = pd.concat(
    [X_base, X_temporal],
    axis=1
)

# Replace infinite values
X_augmented = X_augmented.replace(
    [np.inf, -np.inf], np.nan
)

print("\nOriginal feature count:", X_base.shape[1])
print("New temporal features:", X_temporal.shape[1])
print("Total augmented features:", X_augmented.shape[1])

print("\nNew feature examples:")
print(X_temporal.columns[:15].tolist())

print("\nFinal shape:", X_augmented.shape)


Common phase features: 26

Original feature count: 85
New temporal features: 130
Total augmented features: 215

New feature examples:
['temporal_stress_minus_rest_III_qrs_dur_ms', 'temporal_recovery_minus_stress_III_qrs_dur_ms', 'temporal_recovery_minus_rest_III_qrs_dur_ms', 'temporal_stress_change_pct_III_qrs_dur_ms', 'temporal_recovery_change_pct_III_qrs_dur_ms', 'temporal_stress_minus_rest_III_t_amp_adc', 'temporal_recovery_minus_stress_III_t_amp_adc', 'temporal_recovery_minus_rest_III_t_amp_adc', 'temporal_stress_change_pct_III_t_amp_adc', 'temporal_recovery_change_pct_III_t_amp_adc', 'temporal_stress_minus_rest_II_qrs_dur_ms', 'temporal_recovery_minus_stress_II_qrs_dur_ms', 'temporal_recovery_minus_rest_II_qrs_dur_ms', 'temporal_stress_change_pct_II_qrs_dur_ms', 'temporal_recovery_change_pct_II_qrs_dur_ms']

Final shape: (78, 215)


C:\Users\HP\AppData\Local\Temp\ipykernel_20088\3922187419.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_temporal[f"temporal_stress_minus_rest_{feature}"] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_20088\3922187419.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_temporal[f"temporal_recovery_minus_stress_{feature}"] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_20088\3922187419.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [21]:

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd

# Two experiments
experiments = {
    "Baseline (85 Features)": X_base,
    "Proposed (215 Features)": X_augmented
}

comparison_results = []

cv_compare = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

for experiment_name, X_exp in experiments.items():

    print("\nExperiment:", experiment_name, flush=True)

    for model_name, estimator in models.items():

        oof_prob = np.zeros(len(y_model))

        for train_idx, test_idx in cv_compare.split(
            X_exp, y_model
        ):

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", clone(estimator))
            ])

            pipeline.fit(
                X_exp.iloc[train_idx],
                y_model.iloc[train_idx]
            )

            oof_prob[test_idx] = pipeline.predict_proba(
                X_exp.iloc[test_idx]
            )[:, 1]

        predictions = (oof_prob >= 0.5).astype(int)

        acc = accuracy_score(
            y_model, predictions
        ) * 100

        auc = roc_auc_score(
            y_model, oof_prob
        ) * 100

        comparison_results.append({
            "Experiment": experiment_name,
            "Model": model_name,
            "Accuracy (%)": round(acc, 2),
            "ROC-AUC (%)": round(auc, 2)
        })

        print(
            model_name,
            "| Accuracy:", round(acc, 2),
            "| ROC-AUC:", round(auc, 2),
            flush=True
        )

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.pivot(
    index="Model",
    columns="Experiment",
    values=["Accuracy (%)", "ROC-AUC (%)"]
))

comparison_df.to_csv(
    "MSIMI_Baseline_vs_Proposed_Comparison.csv",
    index=False
)

print("\nBaseline vs Proposed comparison completed!")



Experiment: Baseline (85 Features)
Logistic Regression | Accuracy: 75.64 | ROC-AUC: 42.83
SVM | Accuracy: 88.46 | ROC-AUC: 61.35


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Random Forest | Accuracy: 83.33 | ROC-AUC: 58.29
Kneighbors | Accuracy: 88.46 | ROC-AUC: 51.93
XGBoost | Accuracy: 85.9 | ROC-AUC: 53.62

Experiment: Proposed (215 Features)
Logistic Regression | Accuracy: 73.08 | ROC-AUC: 41.22
SVM | Accuracy: 88.46 | ROC-AUC: 46.38


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Random Forest | Accuracy: 87.18 | ROC-AUC: 57.25
Kneighbors | Accuracy: 88.46 | ROC-AUC: 41.79
XGBoost | Accuracy: 82.05 | ROC-AUC: 48.63


Accuracy (%)                          \
Experiment          Baseline (85 Features) Proposed (215 Features)   
Model                                                                
Kneighbors                           88.46                   88.46   
Logistic Regression                  75.64                   73.08   
Random Forest                        83.33                   87.18   
SVM                                  88.46                   88.46   
XGBoost                              85.90                   82.05   

                               ROC-AUC (%)                          
Experiment          Baseline (85 Features) Proposed (215 Features)  
Model                                                               
Kneighbors                           51.93                   41.79  
Logistic Regression                  42.83                   41.22  
Random Forest                        58.29                   57.25  
SVM                                  61.35                   46.38  
XGBoost                              53.62                   48.63


Baseline vs Proposed comparison completed!


In [22]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)
import numpy as np

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

models_to_check = {
    "Baseline SVM": (X_base, models["SVM"]),
    "Proposed Random Forest": (X_augmented, models["Random Forest"])
}

for name, (X_data, estimator) in models_to_check.items():

    y_true = []
    y_pred = []
    y_score = []

    for train_idx, test_idx in cv.split(X_data, y_model):

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", clone(estimator))
        ])

        pipe.fit(X_data.iloc[train_idx], y_model.iloc[train_idx])

        predictions = pipe.predict(X_data.iloc[test_idx])
        probabilities = pipe.predict_proba(X_data.iloc[test_idx])[:, 1]

        y_true.extend(y_model.iloc[test_idx])
        y_pred.extend(predictions)
        y_score.extend(probabilities)

    print("\n", "=" * 50)
    print(name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))

c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Baseline SVM
Confusion Matrix:
[[69  0]
 [ 9  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        69
           1       0.00      0.00      0.00         9

    accuracy                           0.88        78
   macro avg       0.44      0.50      0.47        78
weighted avg       0.78      0.88      0.83        78

ROC-AUC: 0.6135

Proposed Random Forest
Confusion Matrix:
[[67  2]
 [ 9  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.97      0.92        69
           1       0.00      0.00      0.00         9

    accuracy                           0.86        78
   macro avg       0.44      0.49      0.46        78
weighted avg       0.78      0.86      0.82        78

ROC-AUC: 0.5725


THRESOLD ANALYSIS

In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import confusion_matrix
import numpy as np

X_data = X_base
y_data = y_model

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_prob = np.zeros(len(y_data))

for train_idx, test_idx in cv.split(X_data, y_data):

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clone(models["SVM"]))
    ])

    pipe.fit(
        X_data.iloc[train_idx],
        y_data.iloc[train_idx]
    )

    oof_prob[test_idx] = pipe.predict_proba(
        X_data.iloc[test_idx]
    )[:, 1]

for threshold in [0.50, 0.40, 0.30, 0.20, 0.10]:

    predictions = (oof_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_data, predictions, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    accuracy = (tp + tn) / len(y_data)

    print(f"\nThreshold: {threshold}")
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Sensitivity: {sensitivity:.3f}")
    print(f"Specificity: {specificity:.3f}")
    print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")


Threshold: 0.5
Accuracy: 0.885
Sensitivity: 0.000
Specificity: 1.000
TP: 0, FP: 0, TN: 69, FN: 9

Threshold: 0.4
Accuracy: 0.885
Sensitivity: 0.000
Specificity: 1.000
TP: 0, FP: 0, TN: 69, FN: 9

Threshold: 0.3
Accuracy: 0.885
Sensitivity: 0.000
Specificity: 1.000
TP: 0, FP: 0, TN: 69, FN: 9

Threshold: 0.2
Accuracy: 0.897
Sensitivity: 0.111
Specificity: 1.000
TP: 1, FP: 0, TN: 69, FN: 8

Threshold: 0.1
Accuracy: 0.500
Sensitivity: 0.778
Specificity: 0.464
TP: 7, FP: 37, TN: 32, FN: 2


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Class-weighted SVM

In [24]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)
import numpy as np

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_prob = np.zeros(len(y_model))

for train_idx, test_idx in cv.split(X_base, y_model):

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ])

    pipe.fit(
        X_base.iloc[train_idx],
        y_model.iloc[train_idx]
    )

    oof_prob[test_idx] = pipe.predict_proba(
        X_base.iloc[test_idx]
    )[:, 1]

# Default threshold
predictions = (oof_prob >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_model, predictions))

print("\nClassification Report:")
print(classification_report(
    y_model,
    predictions,
    zero_division=0
))

print("ROC-AUC:", round(roc_auc_score(y_model, oof_prob), 4))
print("PR-AUC:", round(average_precision_score(y_model, oof_prob), 4))

Confusion Matrix:
[[69  0]
 [ 9  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        69
           1       0.00      0.00      0.00         9

    accuracy                           0.88        78
   macro avg       0.44      0.50      0.47        78
weighted avg       0.78      0.88      0.83        78

ROC-AUC: 0.6135
PR-AUC: 0.2848


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Ab hum Random Forest aur SVM par class balancing ke saath hyperparameter tuning aur feature selection try karenge.
Nested Cross-Validation + Hyperparameter Tuning

In [25]:

import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# XGBoost import
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost is not installed. Other models will run.")

# Dataset
X_data = X_base.copy()
y_data = pd.Series(y_model).reset_index(drop=True)
X_data = X_data.reset_index(drop=True)

# Outer CV: unbiased evaluation of the tuned models
outer_cv = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=42
)

# Inner CV: hyperparameter tuning
inner_cv = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=21
)

# Common pipeline
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold()),
        ("select", SelectKBest(score_func=f_classif)),
        ("model", model)
    ])

# Models and parameter grids
experiments = {
    "Logistic Regression": (
        LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        ),
        {
            "select__k": [5, 10, 20, "all"],
            "model__C": [0.01, 0.1, 1, 10]
        }
    ),

    "SVM": (
        SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=False,
            random_state=42
        ),
        {
            "select__k": [5, 10, 20, "all"],
            "model__C": [0.1, 1, 10],
            "model__gamma": ["scale", 0.01]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        {
            "select__k": [5, 10, 20, "all"],
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "select__k": [5, 10, 20, "all"],
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    )
}

if xgb_available:
    experiments["XGBoost"] = (
        XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            n_jobs=1
        ),
        {
            "select__k": [5, 10, 20, "all"],
            "model__n_estimators": [100, 200],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__scale_pos_weight": [1, 3, 7]
        }
    )

results = []

for model_name, (estimator, param_grid) in experiments.items():

    print("\n" + "=" * 60)
    print("MODEL:", model_name)

    oof_predictions = np.zeros(len(y_data), dtype=int)
    oof_scores = np.zeros(len(y_data))

    best_params = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_data, y_data), start=1
    ):

        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]
        y_train = y_data.iloc[train_idx]

        pipeline = make_pipeline(clone(estimator))

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="average_precision",
            cv=inner_cv,
            n_jobs=1,
            refit=True,
            error_score="raise"
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        oof_predictions[test_idx] = best_model.predict(X_test)

        # Ranking score: probability or decision score
        if hasattr(best_model, "predict_proba"):
            scores = best_model.predict_proba(X_test)[:, 1]
        else:
            scores = best_model.decision_function(X_test)

        oof_scores[test_idx] = scores
        best_params.append(search.best_params_)

        print(f"Fold {fold} completed")

    # Overall out-of-fold metrics
    tn, fp, fn, tp = confusion_matrix(
        y_data, oof_predictions, labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(y_data, oof_predictions)
    balanced_acc = balanced_accuracy_score(y_data, oof_predictions)
    precision = precision_score(
        y_data, oof_predictions, zero_division=0
    )
    sensitivity = recall_score(
        y_data, oof_predictions, zero_division=0
    )
    specificity = tn / (tn + fp)
    f1 = f1_score(y_data, oof_predictions, zero_division=0)

    auc = roc_auc_score(y_data, oof_scores)
    ap = average_precision_score(y_data, oof_scores)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy * 100,
        "Balanced Accuracy": balanced_acc * 100,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1 Score": f1,
        "ROC-AUC": auc,
        "PR-AUC": ap,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    })

    print("Confusion Matrix:")
    print(confusion_matrix(y_data, oof_predictions))
    print("Best parameters per fold:")
    for params in best_params:
        print(params)

# Final comparison
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="PR-AUC", ascending=False
)

print("\nFINAL MODEL COMPARISON")
display(results_df.round(4))

results_df.to_csv(
    "MSIMI_Tuned_Models_Comparison.csv",
    index=False
)

print("\nResults saved: MSIMI_Tuned_Models_Comparison.csv")



MODEL: Logistic Regression
Fold 1 completed
Fold 2 completed
Fold 3 completed
Confusion Matrix:
[[55 14]
 [ 6  3]]
Best parameters per fold:
{'model__C': 1, 'select__k': 'all'}
{'model__C': 0.1, 'select__k': 5}
{'model__C': 0.01, 'select__k': 5}

MODEL: SVM


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 1 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 2 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 3 completed
Confusion Matrix:
[[40 29]
 [ 5  4]]
Best parameters per fold:
{'model__C': 10, 'model__gamma': 'scale', 'select__k': 10}
{'model__C': 0.1, 'model__gamma': 0.01, 'select__k': 5}
{'model__C': 1, 'model__gamma': 0.01, 'select__k': 10}

MODEL: Random Forest
Fold 1 completed
Fold 2 completed
Fold 3 completed
Confusion Matrix:
[[64  5]
 [ 8  1]]
Best parameters per fold:
{'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 200, 'select__k': 'all'}
{'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 400, 'select__k': 20}
{'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 200, 'select__k': 5}

MODEL: KNN
Fold 1 completed
Fold 2 completed
Fold 3 completed
Confusion Matrix:
[[67  2]
 [ 9  0]]
Best parameters per fold:
{'model__n_neighbors': 5, 'model__p': 1, 'model__weights': 'distance', 'select__k': 20}
{'model__n_neighbors': 3, 'model__p': 2, 'model__weights': 'distance', 'select__k': 5}
{'mod

,Model,Accuracy,Balanced Accuracy,Precision,Sensitivity,Specificity,F1 Score,ROC-AUC,PR-AUC,TP,FP,TN,FN
1,SVM,56.4103,51.2077,0.1212,0.4444,0.5797,0.1905,0.5733,0.2712,4,29,40,5
2,Random Forest,83.3333,51.9324,0.1667,0.1111,0.9275,0.1333,0.5225,0.1594,1,5,64,8
4,XGBoost,82.0513,51.2077,0.1429,0.1111,0.9130,0.1250,0.5137,0.1469,1,6,63,8
0,Logistic Regression,74.3590,56.5217,0.1765,0.3333,0.7971,0.2308,0.4944,0.1410,3,14,55,6
3,KNN,85.8974,48.5507,0.0000,0.0000,0.9710,0.0000,0.5266,0.1360,0,2,67,9



Results saved: MSIMI_Tuned_Models_Comparison.csv


Repeated Stratified Cross-Validation + SVM, Logistic Regression, KNN, Random Forest aur XGBoost.

In [26]:

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# XGBoost
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost not installed. Skipping XGBoost.")

# Prepare data
X_data = X_base.reset_index(drop=True).copy()
y_data = pd.Series(y_model).reset_index(drop=True)

print("Dataset:", X_data.shape)
print("Positive patients:", int(y_data.sum()))
print("Negative patients:", int((y_data == 0).sum()))

# Repeated outer CV
outer_cv = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=3,
    random_state=42
)

# Inner CV for tuning
inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=21
)

def create_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold()),
        ("select", SelectKBest(score_func=f_classif)),
        ("model", model)
    ])

# Model configurations
experiments = {
    "Logistic Regression": (
        LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        ),
        {
            "select__k": [5, 10, 20],
            "model__C": [0.01, 0.1, 1, 10]
        }
    ),

    "SVM": (
        SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=False
        ),
        {
            "select__k": [5, 10, 20],
            "model__C": [0.1, 1, 10],
            "model__gamma": ["scale", 0.01]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        {
            "select__k": [5, 10, 20],
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 5],
            "model__min_samples_leaf": [1, 2]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "select__k": [5, 10, 20],
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    )
}

if xgb_available:
    experiments["XGBoost"] = (
        XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            n_jobs=1
        ),
        {
            "select__k": [5, 10, 20],
            "model__n_estimators": [100, 200],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__scale_pos_weight": [1, 3, 7]
        }
    )

all_results = []

for model_name, (estimator, param_grid) in experiments.items():

    print("\n" + "=" * 60)
    print("MODEL:", model_name)

    fold_results = []
    all_predictions = []
    all_actual = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_data, y_data), start=1
    ):

        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]

        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]

        search = GridSearchCV(
            estimator=create_pipeline(estimator),
            param_grid=param_grid,
            scoring="average_precision",
            cv=inner_cv,
            n_jobs=1,
            refit=True
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        predictions = best_model.predict(X_test)

        if hasattr(best_model, "predict_proba"):
            scores = best_model.predict_proba(X_test)[:, 1]
        else:
            scores = best_model.decision_function(X_test)

        # Store predictions for pooled confusion matrix
        all_predictions.extend(predictions)
        all_actual.extend(y_test)

        # Calculate AUC separately in each outer fold
        fold_results.append({
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced Accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "Sensitivity": recall_score(
                y_test, predictions, zero_division=0
            ),
            "Specificity": (
                confusion_matrix(
                    y_test, predictions, labels=[0, 1]
                )[0, 0] /
                max(
                    confusion_matrix(
                        y_test, predictions, labels=[0, 1]
                    )[0].sum(), 1
                )
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC-AUC": roc_auc_score(y_test, scores),
            "PR-AUC": average_precision_score(y_test, scores)
        })

        print(f"Fold {fold}/9 completed")

    # Mean and standard deviation across outer folds
    fold_df = pd.DataFrame(fold_results)

    # Pooled confusion matrix: each patient is evaluated
    # once per repetition, so 234 predictions in total.
    cm = confusion_matrix(
        all_actual, all_predictions, labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    summary = {
        "Model": model_name,
        "Accuracy Mean (%)": fold_df["Accuracy"].mean() * 100,
        "Accuracy SD (%)": fold_df["Accuracy"].std() * 100,
        "Balanced Accuracy Mean (%)":
            fold_df["Balanced Accuracy"].mean() * 100,
        "Sensitivity Mean (%)":
            fold_df["Sensitivity"].mean() * 100,
        "Specificity Mean (%)":
            fold_df["Specificity"].mean() * 100,
        "F1 Mean": fold_df["F1"].mean(),
        "ROC-AUC Mean": fold_df["ROC-AUC"].mean(),
        "ROC-AUC SD": fold_df["ROC-AUC"].std(),
        "PR-AUC Mean": fold_df["PR-AUC"].mean(),
        "PR-AUC SD": fold_df["PR-AUC"].std(),
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    }

    all_results.append(summary)

    print("\nPooled Confusion Matrix:")
    print(cm)

# Final table
final_results = pd.DataFrame(all_results)

final_results = final_results.sort_values(
    "PR-AUC Mean", ascending=False
)

print("\nREPEATED CROSS-VALIDATION RESULTS")
display(final_results.round(4))

final_results.to_csv(
    "MSIMI_Repeated_CV_Results.csv",
    index=False
)

print("\nSaved: MSIMI_Repeated_CV_Results.csv")


Dataset: (78, 85)
Positive patients: 9
Negative patients: 69

MODEL: Logistic Regression
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Pooled Confusion Matrix:
[[156  51]
 [ 17  10]]

MODEL: SVM


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 1/9 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 2/9 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

... (11 outputs left out)

c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 9/9 completed

Pooled Confusion Matrix:
[[ 97 110]
 [ 14  13]]

MODEL: Random Forest
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Pooled Confusion Matrix:
[[184  23]
 [ 24   3]]

MODEL: KNN
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Pooled Confusion Matrix:
[[201   6]
 [ 27   0]]

MODEL: XGBoost
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Pooled Confusion Matrix:
[[189  18]
 [ 25   2]]

REPEATED CROSS-VALIDATION RESULTS


,Model,Accuracy Mean (%),Accuracy SD (%),Balanced Accuracy Mean (%),Sensitivity Mean (%),Specificity Mean (%),F1 Mean,ROC-AUC Mean,ROC-AUC SD,PR-AUC Mean,PR-AUC SD,TP,FP,TN,FN
0,Logistic Regression,70.9402,6.4103,56.1997,37.0370,75.3623,0.2105,0.5089,0.1351,0.2385,0.1451,10,51,156,17
3,KNN,85.8974,2.7196,48.5507,0.0000,97.1014,0.0000,0.5298,0.1551,0.2150,0.1166,0,6,201,27
4,XGBoost,81.6239,5.3632,49.3559,7.4074,91.3043,0.0762,0.4742,0.2185,0.2090,0.1219,2,18,189,25
1,SVM,47.0085,29.5915,47.5040,48.1481,46.8599,0.1499,0.4895,0.1067,0.2058,0.1094,13,110,97,14
2,Random Forest,79.9145,3.7378,50.0000,11.1111,88.8889,0.0966,0.5250,0.1216,0.1925,0.0671,3,23,184,24



Saved: MSIMI_Repeated_CV_Results.csv


Bhai, ab hum apne project ke liye ek proper new research idea finalize karte hain. 🔥

Mera proposed idea hai:
Patient-Specific Stress–Recovery Trajectory Learning (PS-SRTL)
Concept: Har patient ke ECG ko uske khud ke Rest baseline ke comparison mein analyze karna, phir Stress ke dauraan changes aur Recovery ke dauraan un changes ke reverse hone ke pattern ko combine karke MSIMI detect karna.

Ye sirf Rest–Stress difference calculate karna nahi hoga. Hum patient ka personal ECG response pattern model karenge.

Bhai, maine research check ki hai aur ek important discovery hui hai: sirf Rest–Stress–Recovery differences use karna new idea nahi hoga. 2024 ke original MSIMI dataset paper mein bhi multistage ECG features aur five ML models use kiye gaye the. 2026 ke ek study mein interstage directional differences bhi investigate kiye gaye hain. 
Scientific Data+1


Isliye humein usse aage jaana hoga.

Hamara proposed research contribution
Proposed new methodology
Baseline-Normalized Stress–Recovery Hysteresis (BNSRH)
Simple meaning: Patient ka ECG stress ke dauraan kitna badla, aur stress khatam hone ke baad uska ECG apne normal baseline ki taraf kitna wapas aaya?







pmc.ncbi.nlm.nih.gov


Ismein hum kya naya karenge?
1. Personal baseline: Har patient ke Rest ECG ko reference banayenge.
2. Normalized stress response: Stress ke changes ko patient ke Rest baseline variability ke respect mein measure karenge.
3. Recovery residual: Recovery ke baad bhi baseline se kitna difference bacha, calculate karenge.
4. Hysteresis indicator: Stress response aur recovery response ke beech asymmetry measure karenge.
5. Combined classifier: In naye features ko original ECG features ke saath train karenge.
Example
Maan lo do patients ka Stress ECG change same hai:
Measurement	Patient A	Patient B
Rest baseline	100	100
Stress	130	130
Recovery	105	125
Recovery residual	5	25

Dono ka stress response same hai, lekin recovery pattern alag hai. Hamara model is difference ko learn karega.
Existing research se difference
Existing approach	Hamara proposed approach
Individual phase features	Patient-normalized response
Rest–Stress difference	Baseline variability-normalized change
Stress–Recovery difference	Recovery residual and asymmetry
Conventional ML comparison	New feature representation + ML

Important: Ye abhi ek research hypothesis hai, globally never-before-implemented technique hone ka verified claim nahi. Personalized ECG monitoring aur adaptive baselines par pehle bhi research hui hai. 
ScienceDirect+1
 Humein systematic literature review aur novelty comparison karna hoga.
Accuracy kaise test karenge?
Hum same patient-wise validation splits par teen experiments compare karenge:
- Model 1: Existing 85 ECG features.
- Model 2: Existing 85 + previous temporal features.
- Model 3: Existing 85 + proposed BNSRH features.
Accuracy, sensitivity, specificity, ROC-AUC aur PR-AUC sab compare honge. Improvement tabhi claim karenge jab validation results support karein.

In [28]:

import pandas as pd
import numpy as np
import re

# Load uploaded feature file
df = pd.read_csv(
    r"C:\Users\HP\Desktop\MSIMI_Project\data\features\08_patient_phase_features.csv"
)

# Use 500 Hz records only
data = df[df["sampling_rate"] == 500].copy()
data = data.reset_index(drop=True)

# Identify common Rest, Stress and Recovery features
prefixes = ["rest_", "stress_", "recover_"]

feature_sets = {
    p: set(c[len(p):] for c in data.columns if c.startswith(p))
    for p in prefixes
}

common = set.intersection(*feature_sets.values())

# Remove ST-related features to avoid target leakage
st_pattern = re.compile(
    r"st60|st80|st_slope|min_st",
    re.IGNORECASE
)

common = sorted([
    f for f in common
    if not st_pattern.search(f)
])

# Remove features that are entirely missing
common = [
    f for f in common
    if data["rest_" + f].notna().any()
    and data["stress_" + f].notna().any()
    and data["recover_" + f].notna().any()
]

print("Common usable features:", len(common))

# Generate BNSRH features
bnsrh = pd.DataFrame()
bnsrh["subject_id"] = data["subject_id"]

for f in common:

    rest = data["rest_" + f]
    stress = data["stress_" + f]
    recovery = data["recover_" + f]

    # Stress response from personal Rest baseline
    stress_change = stress - rest

    # Residual deviation after Recovery
    recovery_residual = recovery - rest

    # Difference between stress and recovery response
    hysteresis = stress - recovery

    # Recovery fraction; undefined when stress change is near zero
    recovery_fraction = (
        1 - recovery_residual.abs() /
        stress_change.abs().replace(0, np.nan)
    )

    bnsrh[f + "_stress_change"] = stress_change
    bnsrh[f + "_recovery_residual"] = recovery_residual
    bnsrh[f + "_hysteresis"] = hysteresis
    bnsrh[f + "_recovery_fraction"] = recovery_fraction

print("BNSRH shape:", bnsrh.shape)
display(bnsrh.head())

# Save engineered features
bnsrh.to_csv(
    "MSIMI_BNSRH_Features_500Hz.csv",
    index=False
)

print("BNSRH features saved successfully!")


Common usable features: 26
BNSRH shape: (80, 105)


C:\Users\HP\AppData\Local\Temp\ipykernel_20088\1893407366.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bnsrh[f + "_recovery_fraction"] = recovery_fraction
C:\Users\HP\AppData\Local\Temp\ipykernel_20088\1893407366.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bnsrh[f + "_stress_change"] = stress_change
C:\Users\HP\AppData\Local\Temp\ipykernel_20088\1893407366.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

,subject_id,III_qrs_dur_ms_stress_change,III_qrs_dur_ms_recovery_residual,III_qrs_dur_ms_hysteresis,III_qrs_dur_ms_recovery_fraction,III_t_amp_adc_stress_change,III_t_amp_adc_recovery_residual,III_t_amp_adc_hysteresis,III_t_amp_adc_recovery_fraction,II_qrs_dur_ms_stress_change,...,aVR_t_amp_adc_hysteresis,aVR_t_amp_adc_recovery_fraction,qrs_axis_degrees_stress_change,qrs_axis_degrees_recovery_residual,qrs_axis_degrees_hysteresis,qrs_axis_degrees_recovery_fraction,raw_detector_rr_rmssd_ms_stress_change,raw_detector_rr_rmssd_ms_recovery_residual,raw_detector_rr_rmssd_ms_hysteresis,raw_detector_rr_rmssd_ms_recovery_fraction
0,sub_001,-2.0,-4.0,2.0,-1.000000,12.480,11.669,0.811,6.498397e-02,-2.0,...,0.738,-0.342142,-8.11,-9.57,1.46,-0.180025,-54.15,-70.43,16.28,-0.300646
1,sub_002,32.0,12.0,20.0,0.625000,4.135,-4.135,8.270,-6.661338e-16,12.0,...,33.778,-5.604447,-3.33,-4.84,1.51,-0.453453,-7.17,-24.26,17.09,-2.383543
2,sub_003,68.0,-10.0,78.0,0.852941,184.379,6.910,177.469,9.625228e-01,4.0,...,-132.135,0.976427,12.19,5.34,6.85,0.561936,-61.51,-42.83,-18.68,0.303690
3,sub_004,4.0,6.0,-2.0,-0.500000,-107.565,-105.715,-1.850,1.719890e-02,-74.0,...,0.759,0.004513,-11.57,-9.84,-1.73,0.149525,-2.49,-16.15,13.66,-5.485944
4,sub_005,4.0,-6.0,10.0,-0.500000,-5.233,-1.721,-3.512,6.711255e-01,-16.0,...,-0.501,-0.018700,17.49,5.84,11.65,0.666095,19.26,-17.35,36.61,0.099169


BNSRH features saved successfully!


In [29]:

import pandas as pd
import numpy as np

# Existing labelled patient IDs
labelled_ids = data.loc[
    data["is_unlabelled_holdout"] == False,
    "subject_id"
].tolist()

# Check alignment
print("X_base:", X_base.shape)
print("y_model:", len(y_model))
print("Labelled patients:", len(labelled_ids))

assert len(X_base) == len(y_model) == len(labelled_ids), (
    "Patient rows and labels are not aligned. Stop and check."
)

# Arrange BNSRH features in the same patient order
bnsrh_aligned = (
    bnsrh.set_index("subject_id")
    .reindex(labelled_ids)
)

# Exclude unstable recovery fraction features
bnsrh_features = bnsrh_aligned.drop(
    columns=[
        c for c in bnsrh_aligned.columns
        if c.endswith("_recovery_fraction")
    ]
)

# Combine with existing baseline features
X_original = X_base.reset_index(drop=True).copy()

X_proposed = pd.concat(
    [
        X_original,
        bnsrh_features.reset_index(drop=True)
    ],
    axis=1
)

y = pd.Series(y_model).reset_index(drop=True)

print("\nOriginal features:", X_original.shape)
print("BNSRH features:", bnsrh_features.shape)
print("Proposed features:", X_proposed.shape)
print("Labels:", y.shape)

assert not X_proposed.columns.duplicated().any()

print("\nBNSRH integration successful!")


X_base: (78, 85)
y_model: 78
Labelled patients: 78

Original features: (78, 85)
BNSRH features: (78, 78)
Proposed features: (78, 163)
Labels: (78,)

BNSRH integration successful!


In [30]:

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score
)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=3000
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(
        kernel="rbf", class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    )
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.05,
        scale_pos_weight=3,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )
except ImportError:
    print("XGBoost unavailable; other models will run.")

cv = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=3,
    random_state=42
)

datasets = {
    "Original": X_original,
    "Original + BNSRH": X_proposed
}

results = []

for model_name, estimator in models.items():

    for dataset_name, X in datasets.items():

        fold_metrics = []

        for train_idx, test_idx in cv.split(X, y):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            k = min(20, X.shape[1])

            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("variance", VarianceThreshold()),
                ("select", SelectKBest(f_classif, k=k)),
                ("model", estimator)
            ])

            pipe.fit(X_train, y_train)

            pred = pipe.predict(X_test)

            if hasattr(pipe, "predict_proba"):
                score = pipe.predict_proba(X_test)[:, 1]
            else:
                score = pipe.decision_function(X_test)

            fold_metrics.append({
                "Accuracy": accuracy_score(y_test, pred),
                "Balanced Accuracy": balanced_accuracy_score(
                    y_test, pred
                ),
                "Sensitivity": recall_score(
                    y_test, pred, zero_division=0
                ),
                "Precision": precision_score(
                    y_test, pred, zero_division=0
                ),
                "F1": f1_score(y_test, pred, zero_division=0),
                "ROC-AUC": roc_auc_score(y_test, score),
                "PR-AUC": average_precision_score(y_test, score)
            })

        avg = pd.DataFrame(fold_metrics).mean()

        results.append({
            "Model": model_name,
            "Features": dataset_name,
            **avg.to_dict()
        })

        print("Completed:", model_name, "|", dataset_name)

comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    ["Features", "PR-AUC"],
    ascending=[True, False]
)

display(comparison.round(4))

comparison.to_csv(
    "BNSRH_Model_Comparison.csv",
    index=False
)

print("Comparison saved successfully!")


Completed: Logistic Regression | Original
Completed: Logistic Regression | Original + BNSRH
Completed: KNN | Original
Completed: KNN | Original + BNSRH
Completed: SVM | Original
Completed: SVM | Original + BNSRH
Completed: Random Forest | Original
Completed: Random Forest | Original + BNSRH
Completed: XGBoost | Original
Completed: XGBoost | Original + BNSRH


,Model,Features,Accuracy,Balanced Accuracy,Sensitivity,Precision,F1,ROC-AUC,PR-AUC
8,XGBoost,Original,0.8504,0.5129,0.0741,0.0926,0.0815,0.5266,0.2494
6,Random Forest,Original,0.8419,0.5081,0.0741,0.0926,0.0815,0.5185,0.2065
4,SVM,Original,0.8120,0.4911,0.0741,0.0648,0.0688,0.4332,0.1852
0,Logistic Regression,Original,0.6368,0.4243,0.1481,0.0667,0.0906,0.4300,0.1785
2,KNN,Original,0.8590,0.5016,0.0370,0.1111,0.0556,0.5314,0.1711
5,SVM,Original + BNSRH,0.8504,0.4807,0.0000,0.0000,0.0000,0.5572,0.3000
7,Random Forest,Original + BNSRH,0.8376,0.4895,0.0370,0.0278,0.0317,0.5306,0.2113
3,KNN,Original + BNSRH,0.8846,0.5000,0.0000,0.0000,0.0000,0.5821,0.1961
1,Logistic Regression,Original + BNSRH,0.6880,0.5177,0.2963,0.1365,0.1828,0.4702,0.1913
9,XGBoost,Original + BNSRH,0.8462,0.5105,0.0741,0.0926,0.0815,0.5201,0.1911


Comparison saved successfully!


Training-Only Threshold Optimization + BNSRH Validation.

In [31]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# 1. Models
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ),

    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    )
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        scale_pos_weight=3,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )
except ImportError:
    print("XGBoost unavailable.")

# 2. Same repeated patient-wise CV
outer_cv = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=3,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=21
)

datasets = {
    "Original": X_original.reset_index(drop=True),
    "Original + BNSRH": X_proposed.reset_index(drop=True)
}

y = pd.Series(y).reset_index(drop=True)

# 3. Pipeline
def make_pipeline(model, feature_count):

    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold()),
        ("select", SelectKBest(
            score_func=f_classif,
            k=min(20, feature_count)
        )),
        ("model", clone(model))
    ])

# 4. Select threshold using training-only predictions
def choose_threshold(y_true, probabilities):

    thresholds = np.linspace(0.05, 0.95, 91)

    best_threshold = 0.50
    best_balanced_accuracy = -1
    best_sensitivity = -1

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        bal_acc = balanced_accuracy_score(
            y_true, predictions
        )

        sensitivity = recall_score(
            y_true, predictions, zero_division=0
        )

        if (
            bal_acc > best_balanced_accuracy
            or (
                np.isclose(bal_acc, best_balanced_accuracy)
                and sensitivity > best_sensitivity
            )
        ):
            best_balanced_accuracy = bal_acc
            best_sensitivity = sensitivity
            best_threshold = threshold

    return best_threshold

# 5. Run experiment
all_results = []

for model_name, model in models.items():

    for dataset_name, X in datasets.items():

        print("\nRunning:", model_name, "|", dataset_name)

        fold_results = []

        for fold, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y), start=1
        ):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            # Base pipeline
            base_pipeline = make_pipeline(
                model, X.shape[1]
            )

            # Calibrated classifier
            calibrated_model = CalibratedClassifierCV(
                estimator=base_pipeline,
                method="sigmoid",
                cv=inner_cv,
                ensemble=False
            )

            # Training-only out-of-fold probabilities
            train_oof_probabilities = cross_val_predict(
                calibrated_model,
                X_train,
                y_train,
                cv=inner_cv,
                method="predict_proba",
                n_jobs=1
            )[:, 1]

            # Select threshold without seeing test data
            best_threshold = choose_threshold(
                y_train,
                train_oof_probabilities
            )

            # Fit on complete outer training data
            calibrated_model.fit(X_train, y_train)

            # Evaluate on unseen outer test data
            test_probabilities = calibrated_model.predict_proba(
                X_test
            )[:, 1]

            tuned_predictions = (
                test_probabilities >= best_threshold
            ).astype(int)

            default_predictions = (
                test_probabilities >= 0.50
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                y_test,
                tuned_predictions,
                labels=[0, 1]
            ).ravel()

            fold_results.append({
                "Threshold": best_threshold,

                "Accuracy": accuracy_score(
                    y_test, tuned_predictions
                ),

                "Accuracy_0.5": accuracy_score(
                    y_test, default_predictions
                ),

                "Balanced_Accuracy": balanced_accuracy_score(
                    y_test, tuned_predictions
                ),

                "Sensitivity": recall_score(
                    y_test, tuned_predictions,
                    zero_division=0
                ),

                "Specificity": tn / max(tn + fp, 1),

                "Precision": precision_score(
                    y_test, tuned_predictions,
                    zero_division=0
                ),

                "F1": f1_score(
                    y_test, tuned_predictions,
                    zero_division=0
                ),

                "ROC_AUC": roc_auc_score(
                    y_test, test_probabilities
                ),

                "PR_AUC": average_precision_score(
                    y_test, test_probabilities
                )
            })

            print(f"Fold {fold}/9 completed")

        fold_df = pd.DataFrame(fold_results)

        result = {
            "Model": model_name,
            "Features": dataset_name,
        }

        for metric in fold_df.columns:
            result[metric + "_Mean"] = fold_df[metric].mean()
            result[metric + "_SD"] = fold_df[metric].std()

        all_results.append(result)

# 6. Final results
threshold_results = pd.DataFrame(all_results)

display(
    threshold_results.round(4)
)

threshold_results.to_csv(
    "BNSRH_Threshold_Optimization_Results.csv",
    index=False
)

print("\nResults saved successfully!")



Running: Logistic Regression | Original
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: Logistic Regression | Original + BNSRH
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: KNN | Original
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: KNN | Original + BNSRH
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: SVM | Original


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 1/9 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 2/9 completed


c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

... (29 outputs left out)

c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\anaconda3\envs\ds311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 

Fold 9/9 completed

Running: Random Forest | Original
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: Random Forest | Original + BNSRH
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: XGBoost | Original
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed

Running: XGBoost | Original + BNSRH
Fold 1/9 completed
Fold 2/9 completed
Fold 3/9 completed
Fold 4/9 completed
Fold 5/9 completed
Fold 6/9 completed
Fold 7/9 completed
Fold 8/9 completed
Fold 9/9 completed


,Model,Features,Threshold_Mean,Threshold_SD,Accuracy_Mean,Accuracy_SD,Accuracy_0.5_Mean,Accuracy_0.5_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,...,Specificity_Mean,Specificity_SD,Precision_Mean,Precision_SD,F1_Mean,F1_SD,ROC_AUC_Mean,ROC_AUC_SD,PR_AUC_Mean,PR_AUC_SD
0,Logistic Regression,Original,0.1678,0.1628,0.5385,0.3370,0.8846,0.0000,0.5298,0.1184,...,0.5411,0.4376,0.0829,0.0926,0.1405,0.1504,0.5427,0.1751,0.2534,0.1564
1,Logistic Regression,Original + BNSRH,0.1478,0.0738,0.5385,0.3699,0.8846,0.0000,0.4976,0.0413,...,0.5507,0.4861,0.0536,0.0639,0.0956,0.1139,0.4171,0.0933,0.1635,0.0555
2,KNN,Original,0.1700,0.1593,0.4744,0.3022,0.8761,0.0256,0.4614,0.0763,...,0.4783,0.3858,0.0786,0.0509,0.1294,0.0831,0.4251,0.0870,0.1233,0.0288
3,KNN,Original + BNSRH,0.1478,0.0919,0.6282,0.2852,0.8846,0.0000,0.4678,0.0618,...,0.6763,0.3619,0.0527,0.0536,0.0851,0.0861,0.4283,0.1118,0.1621,0.1062
4,SVM,Original,0.1167,0.0660,0.4060,0.2879,0.8846,0.0000,0.4710,0.0665,...,0.3865,0.3705,0.0958,0.0458,0.1571,0.0708,0.3977,0.1144,0.1796,0.0930
5,SVM,Original + BNSRH,0.1578,0.0814,0.5769,0.3342,0.8846,0.0000,0.4710,0.0394,...,0.6087,0.4342,0.0453,0.0552,0.0793,0.0975,0.3977,0.1459,0.1826,0.1057
6,Random Forest,Original,0.1333,0.0658,0.4701,0.2614,0.8846,0.0000,0.4428,0.1017,...,0.4783,0.3340,0.0899,0.0747,0.1358,0.0947,0.4469,0.0831,0.1572,0.0295
7,Random Forest,Original + BNSRH,0.1889,0.1328,0.5641,0.3846,0.8846,0.0000,0.4960,0.0121,...,0.5845,0.4991,0.0502,0.0596,0.0892,0.1060,0.4911,0.1385,0.2202,0.1293
8,XGBoost,Original,0.1200,0.0332,0.5342,0.3215,0.8803,0.0128,0.4630,0.0561,...,0.5556,0.4163,0.0622,0.0642,0.1015,0.1010,0.4678,0.2071,0.2238,0.1320
9,XGBoost,Original + BNSRH,0.1189,0.0262,0.5897,0.3125,0.8846,0.0000,0.4944,0.0995,...,0.6184,0.4100,0.0627,0.0765,0.1024,0.1236,0.4155,0.1554,0.1567,0.0654



Results saved successfully!
